# 07 - Controlled Comparison of Defect Configurations

## Aim of this notebook

In the previous notebook, we performed constrained random optimization to search for high-scoring hydrogen-like defect configurations. However, random search mixes several effects, including defect number, defect strength, spatial arrangement, and random sampling.

In this notebook, we perform controlled comparison experiments. We define representative defect cases such as clean, weak spread, strong spread, weak clustered, strong clustered, random, and the best optimized configuration from Notebook 6.

For each case, we compute the energy spectrum, localization descriptors, and transport-related score.

The goal is to understand how defect strength and spatial arrangement influence localization and the designed transport descriptor under controlled conditions.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.linalg import eigh

ROOT = Path('..').resolve()
RESULTS_DIR = ROOT / 'results'
FIGURE_DIR = ROOT / 'figures'

RESULTS_DIR.mkdir(exist_ok=True)
FIGURE_DIR.mkdir(exist_ok=True)

## Reuse Physics Functions

These functions are kept consistent with the earlier notebooks.

In [ ]:
def build_clean_hamiltonian(n_sites=40, hopping=1.0, epsilon_0=0.0):
    """Build a clean 1D nearest-neighbor tight-binding Hamiltonian."""
    hamiltonian = np.zeros((n_sites, n_sites), dtype=float)
    np.fill_diagonal(hamiltonian, epsilon_0)

    for i in range(n_sites - 1):
        hamiltonian[i, i + 1] = -hopping
        hamiltonian[i + 1, i] = -hopping

    return hamiltonian


def add_defects_at_sites(hamiltonian, defect_sites, defect_strength=2.0):
    """Add hydrogen-like defects as local diagonal perturbations."""
    defected = hamiltonian.copy()

    for site in defect_sites:
        defected[int(site), int(site)] += defect_strength

    return defected


def compute_ipr(eigenvectors):
    """Compute inverse participation ratio for each eigenstate."""
    return np.sum(np.abs(eigenvectors) ** 4, axis=0)


def defect_density(n_defects, n_sites):
    """Calculate defect density."""
    return n_defects / n_sites


def mean_defect_distance(defect_sites):
    """Calculate the mean distance between neighboring defect sites."""
    defect_sites = sorted(defect_sites)

    if len(defect_sites) < 2:
        return np.nan

    return float(np.mean(np.diff(defect_sites)))


def clustering_index(defect_sites):
    """Calculate clustering index. Undefined for fewer than two defects."""
    mean_distance = mean_defect_distance(defect_sites)

    if np.isnan(mean_distance):
        return np.nan

    return 1.0 / (1.0 + mean_distance)


def energy_gap(eigenvalues):
    """Calculate central energy gap."""
    n = len(eigenvalues)
    return float(eigenvalues[n // 2] - eigenvalues[n // 2 - 1])


def transport_descriptor(mean_ipr, density, alpha=1.0, beta=1.0):
    """Calculate transport-related descriptor."""
    return 1.0 / (1.0 + alpha * mean_ipr + beta * density)

## Define Controlled Cases

Most defected cases use `n_defects = 3`, so defect density is fixed at `3 / 40 = 0.075`. This helps separate arrangement and strength effects.

In [ ]:
n_sites = 40
hopping = 1.0
epsilon_0 = 0.0
n_defects = 3
weak_strength = 0.5
strong_strength = 3.5

spread_sites = [0, 19, 39]
clustered_sites = [18, 19, 20]

rng = np.random.default_rng(42)
random_sites = sorted(rng.choice(n_sites, size=n_defects, replace=False).tolist())

comparison_cases = [
    {
        'case': 'Clean system',
        'arrangement_mode': 'clean',
        'defect_sites': [],
        'defect_strength': 0.0,
    },
    {
        'case': 'Weak spread defects',
        'arrangement_mode': 'spread',
        'defect_sites': spread_sites,
        'defect_strength': weak_strength,
    },
    {
        'case': 'Strong spread defects',
        'arrangement_mode': 'spread',
        'defect_sites': spread_sites,
        'defect_strength': strong_strength,
    },
    {
        'case': 'Weak clustered defects',
        'arrangement_mode': 'clustered',
        'defect_sites': clustered_sites,
        'defect_strength': weak_strength,
    },
    {
        'case': 'Strong clustered defects',
        'arrangement_mode': 'clustered',
        'defect_sites': clustered_sites,
        'defect_strength': strong_strength,
    },
    {
        'case': 'Random defects',
        'arrangement_mode': 'random',
        'defect_sites': random_sites,
        'defect_strength': 2.0,
    },
]

best_config_path = RESULTS_DIR / 'notebook6_best_configuration.csv'
if best_config_path.exists():
    best_config = pd.read_csv(best_config_path).iloc[0]
    comparison_cases.append(
        {
            'case': 'Best constrained case',
            'arrangement_mode': str(best_config['arrangement_mode']),
            'defect_sites': [int(site) for site in str(best_config['defect_sites']).split(';') if site != ''],
            'defect_strength': float(best_config['defect_strength']),
        }
    )

comparison_cases

## Evaluate Each Case

For each case, we compute the energy spectrum, IPR values, spectral descriptors, and transport-related score.

In [ ]:
def evaluate_case(case_definition):
    H_clean = build_clean_hamiltonian(
        n_sites=n_sites,
        hopping=hopping,
        epsilon_0=epsilon_0,
    )

    defect_sites = case_definition['defect_sites']
    defect_strength = case_definition['defect_strength']

    if len(defect_sites) == 0:
        H_case = H_clean
    else:
        H_case = add_defects_at_sites(
            H_clean,
            defect_sites=defect_sites,
            defect_strength=defect_strength,
        )

    eigenvalues, eigenvectors = eigh(H_case)
    ipr_values = compute_ipr(eigenvectors)
    n_case_defects = len(defect_sites)
    density = defect_density(n_case_defects, n_sites)
    mean_ipr = float(np.mean(ipr_values))

    return {
        'case': case_definition['case'],
        'arrangement_mode': case_definition['arrangement_mode'],
        'n_sites': n_sites,
        'n_defects': n_case_defects,
        'defect_density': density,
        'defect_strength': defect_strength,
        'defect_sites': ';'.join(map(str, defect_sites)),
        'mean_defect_distance': mean_defect_distance(defect_sites),
        'clustering_index': clustering_index(defect_sites),
        'energy_gap': energy_gap(eigenvalues),
        'energy_bandwidth': float(np.max(eigenvalues) - np.min(eigenvalues)),
        'mean_ipr': mean_ipr,
        'max_ipr': float(np.max(ipr_values)),
        'transport_descriptor': transport_descriptor(mean_ipr, density),
        'eigenvalues': eigenvalues,
        'ipr_values': ipr_values,
    }


case_results = [evaluate_case(case) for case in comparison_cases]

comparison_table = pd.DataFrame(
    [
        {key: value for key, value in result.items() if key not in {'eigenvalues', 'ipr_values'}}
        for result in case_results
    ]
)

comparison_table.to_csv(RESULTS_DIR / 'notebook7_comparison_cases.csv', index=False)
comparison_table

## Energy Spectra Comparison

Defects modify the energy spectrum, especially when defect strength is high.

In [ ]:
spectra_cases = {
    'Clean system',
    'Weak spread defects',
    'Strong spread defects',
    'Strong clustered defects',
    'Best constrained case',
}

plt.figure(figsize=(9, 5))
for result in case_results:
    if result['case'] in spectra_cases:
        plt.plot(
            result['eigenvalues'],
            marker='o',
            markersize=3,
            linewidth=1,
            label=result['case'],
        )

plt.xlabel('Eigenstate index')
plt.ylabel('Energy')
plt.title('Energy spectra for controlled comparison cases')
plt.legend(fontsize=8)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'notebook7_energy_spectra_comparison.png', dpi=200)
plt.show()

## Localization Comparison

The mean IPR summarizes localization across eigenstates. Larger values indicate stronger localization.

In [ ]:
plt.figure(figsize=(9, 5))
plt.bar(comparison_table['case'], comparison_table['mean_ipr'])
plt.ylabel('Mean IPR')
plt.title('Mean localization by comparison case')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'notebook7_mean_ipr_comparison.png', dpi=200)
plt.show()

## Transport Descriptor Comparison

Configurations with lower localization and lower defect density tend to have higher transport-related scores.

In [ ]:
plt.figure(figsize=(9, 5))
plt.bar(comparison_table['case'], comparison_table['transport_descriptor'], color='tab:green')
plt.ylabel('Transport descriptor')
plt.title('Transport descriptor by comparison case')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'notebook7_transport_descriptor_comparison.png', dpi=200)
plt.show()

## Select Candidate Cases for Notebook 8

Notebook 8 will start Qiskit validation. For that step, we select a small set of representative cases.

In [ ]:
candidate_case_names = [
    'Clean system',
    'Weak spread defects',
    'Strong clustered defects',
    'Best constrained case',
]

candidate_cases = comparison_table[
    comparison_table['case'].isin(candidate_case_names)
].copy()

candidate_cases.to_csv(RESULTS_DIR / 'notebook7_candidate_cases_for_qiskit.csv', index=False)
candidate_cases

## Conclusion

In this notebook, we compared representative defect configurations under controlled conditions. Unlike the random optimization experiment, this comparison fixed or explicitly selected the defect number, defect strength, and spatial arrangement.

The results allow us to interpret the physical effect of defect configuration more clearly. Weak and spread defect configurations are expected to produce lower localization and higher transport-related scores, while strong or clustered defects may increase localization and reduce the transport proxy.

This controlled comparison supports the interpretation that defect strength and spatial arrangement both influence localization and transport-related behavior in the model system.

Notebook 8 can now choose a small representative case, reduce the Hamiltonian size, and begin Qiskit-based validation.